# How hyperparameters shape a deep network: colorectal cancer tissue classification on PathMNIST

**MS Health Data Science capstone · Aga Khan University · Israr Hussain, Sumreen Fatima**
Code: [github.com/bitsbyte111/pathmnist-crc-classification](https://github.com/bitsbyte111/pathmnist-crc-classification)

The project applies the concepts of our machine-learning and deep-learning courses to a real clinical image task: classifying H&E-stained colorectal tissue patches into 9 tissue types. The aim is to *understand* what each design choice does, not to chase the last 0.1 % of accuracy. Every experiment states a hypothesis, measures the outcome on validation data, and records an observation.

| Study | What varies | Concepts |
|---|---|---|
| **1. CNN from scratch** (28 px, 18 stages) | optimiser, learning rate, batch size, LR schedule, activation, initialisation, kernel size, width, depth, convolutions per block, pooling, classifier head, batch norm, dropout, weight decay, augmentation, label smoothing, class weights | gradient descent variants, vanishing gradients, receptive field, capacity vs over-fitting, regularisation |
| **2. Transfer learning** (ResNet-50, 224 px, 10 stages) | fine-tuning depth, optimiser, learning rate, batch size, warm-up, input normalisation, dropout, weight decay, augmentation, label smoothing | feature transfer, catastrophic forgetting, domain shift |
| **3. Joint search** (Optuna, optional) | learning rate, weight decay, batch size, dropout, label smoothing together | interactions missed by one-at-a-time search |
| **4. Benchmark** (test cohort) | MLP, CNNs, ResNet-18/50, DenseNet, EfficientNet, ConvNeXt, DeiT, Swin, MLP-Mixer | generalisation to unseen patients, statistical comparison |
| **5. Explainability** | SHAP (pixel and H&E stain space), Grad-CAM | what the model looks at |

**Leakage control.** Hyperparameters are chosen on the validation split only. The test split (CRC-VAL-HE-7K, 50 patients not seen in training) is used once, in Study 4.

### Running it
**Settings:** *Accelerator* → **GPU T4 ×2**; *Internet* → **On**; *Add Input* → your PathMNIST dataset (and, from the 2nd version on, this notebook's previous output).

Turn on the studies for this version in the next cell, then **Save Version → Save & Run All (Commit)**. Suggested sequence, each within Kaggle's 12-hour limit (times are estimates):

| Version | Switches | Approx. time on T4 ×2 |
|---|---|---|
| 1 | `RUN_CHECKS`, `RUN_CNN_STUDY`, `RUN_B2` | 3–4 h |
| 2 | `RUN_TRANSFER_STUDY`, `RUN_OPTUNA` | 2–4 h |
| 3 | `RUN_BENCHMARK`, `BENCHMARK_ONLY` = the scratch models + P1–P3 | ≤ 10 h |
| 4 | `RUN_BENCHMARK`, `BENCHMARK_ONLY` = P4–P7 | ≤ 10 h |
| 5 | `RUN_BENCHMARK` (all, already finished, so only the report), `RUN_EXPLAIN` | ~1 h |

## 1. Configuration

In [ ]:
# ---- what this version runs ------------------------------------------------------------
RUN_CHECKS         = True    # tests on synthetic data (~2 min)
RUN_CNN_STUDY      = False   # Study 1: 18-stage CNN hyperparameter study (28 px)
RUN_B2             = False   # MedMNIST v2 ResNet-18 reproduction (pipeline validation)
RUN_TRANSFER_STUDY = False   # Study 2: transfer-learning hyperparameter study (224 px)
TRANSFER_BASE      = "configs/models/p1_resnet50.yaml"   # or "configs/models/p5_deit_small.yaml" (transformer)
RUN_OPTUNA         = False   # Study 3: joint Optuna search around the tuned CNN (needs Study 1)
RUN_BENCHMARK      = False   # Study 4: final models on the test cohort
BENCHMARK_ONLY     = []      # e.g. ["P4_convnext_tiny", "P5_deit_small"]; [] = every model
RUN_EXPLAIN        = False   # Study 5: SHAP + Grad-CAM for the best model
RUN_E4, RUN_E5     = False, False   # colour-augmentation and resolution studies (secondary)

# ---- settings -----------------------------------------------------------------------------
SIZES        = [28, 224]                 # resolutions to prepare
DOWNLOAD_MISSING = True                  # fetch any size not found in /kaggle/input from Zenodo (official, checksum-verified)
SEEDS        = None                      # e.g. [42, 43] to repeat Study 1 with two seeds; None = plan default
REPO_URL     = "https://github.com/bitsbyte111/pathmnist-crc-classification.git"
CODE_DATASET = None                      # "/kaggle/input/<code-dataset>" if the code is attached as a dataset
GPUS         = "auto"                    # "auto" = one run per GPU; "0" = single GPU

## 2. Environment and GPU monitor

In [ ]:
import os, sys, json, re, shutil, subprocess
from pathlib import Path
import torch
assert torch.cuda.is_available(), "No GPU: set Accelerator to 'GPU T4 x2' in the notebook settings"
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 2**30:.0f} GB")
!nproc && free -g | head -2

# Log GPU utilisation and memory every 30 s for the whole version (plotted at the end)
GPU_LOG = "/kaggle/working/gpu_usage.csv"
gpu_monitor = subprocess.Popen(
    f"nvidia-smi --query-gpu=timestamp,index,utilization.gpu,memory.used --format=csv,noheader,nounits -l 30 >> {GPU_LOG}",
    shell=True)

def free_gb(p):
    try:
        os.makedirs(p, exist_ok=True); return shutil.disk_usage(p).free / 1e9
    except OSError:
        return 0.0
DATA_ROOT = max(["/kaggle/temp/crc_data", "/tmp/crc_data"], key=free_gb)   # scratch disk for the data cache
PLATFORM = f"--platform kaggle -o data.root={DATA_ROOT}"
print(f"data cache: {DATA_ROOT} ({free_gb(DATA_ROOT):.0f} GB free)")

## 3. Code

In [ ]:
WORK = Path("/kaggle/working/crc-tissuenet")
if CODE_DATASET:
    shutil.copytree(next(Path(CODE_DATASET).rglob("pyproject.toml")).parent, WORK, dirs_exist_ok=True)
elif not WORK.exists():
    !git clone -q {REPO_URL} {WORK}
os.chdir(WORK)
!pip install -q --no-deps --no-build-isolation -e .
import importlib.util
missing = [m for m in ("shap", "optuna", "tabulate", "timm") if importlib.util.find_spec(m) is None]
if missing:
    !pip install -q {" ".join(missing)}
sys.path.insert(0, str(WORK / "src"))
import timm, shap
print("torch", torch.__version__, "| timm", timm.__version__, "| shap", shap.__version__)

## 4. Resume from previous versions
Attach earlier outputs via *Add Input → Your Work*. Finished runs and tuned configurations are copied back and never re-trained. Outputs of a teammate's copy of this notebook can be attached too.

In [ ]:
for prev in Path("/kaggle/input").glob("*/crc-tissuenet"):
    if (prev / "runs").is_dir():
        print("restoring runs from", prev)
        shutil.copytree(prev / "runs", WORK / "runs", dirs_exist_ok=True)
    for tuned in (prev / "configs/models").glob("*_tuned.yaml"):
        if not (WORK / "configs/models" / tuned.name).exists():
            shutil.copy(tuned, WORK / "configs/models" / tuned.name); print("restored", tuned.name)
n = len(list((WORK / "runs").glob("**/metrics.json"))) if (WORK / "runs").exists() else 0
print("finished runs available:", n)

## 5. Data
PathMNIST (MedMNIST v2/+; CC BY 4.0) is derived from NCT-CRC-HE-100K and CRC-VAL-HE-7K. The training and validation splits come from 86 slides; the test split from 50 other patients.

The loader first looks for PathMNIST files in `/kaggle/input`, for example a 28 px Kaggle dataset you attached. With `DOWNLOAD_MISSING = True`, any size it does not find (usually 224 px) is downloaded directly from the official MedMNIST record on Zenodo and verified by checksum. Everything stays inside Kaggle.

The data are cached as memory-mapped arrays on scratch disk, which is not saved between versions. **Tip:** export the data once with `scripts/export_npy.py` in a CPU-only notebook and attach its output (see `docs/KAGGLE.md`). Later versions then skip both the download and the decompression.

In [ ]:
ROOT_ARG = PLATFORM.replace("-o data.root=", "--root ")
DL = "--download" if DOWNLOAD_MISSING else ""
!python scripts/prepare_data.py {ROOT_ARG} --sizes {" ".join(map(str, SIZES))} {DL}

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
from crc_tissuenet.data import CLASS_NAMES, CLASS_DESCRIPTIONS, load_split, prepare_data

summ = json.load(open(f"{DATA_ROOT}/pathmnist_28_summary.json"))
dist = pd.DataFrame({s: summ[s]["class_counts"] for s in ("train", "val", "test")})
fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
(dist / dist.sum()).plot.bar(ax=ax[0]); ax[0].set(title="Class proportions per split", ylabel="share")
d28 = prepare_data(DATA_ROOT, 28, download=False, source_dirs=["/kaggle/input"])
rng = np.random.default_rng(0)
for split, color in (("train", "tab:blue"), ("test", "tab:red")):
    a = load_split(d28, split, in_memory=False)
    x = np.asarray(a.images[np.sort(rng.choice(len(a), 3000, replace=False))], dtype=float) / 255
    for c, ls in zip(range(3), ("-", "--", ":")):
        ax[1].hist(x[..., c].ravel(), bins=60, histtype="step", density=True, color=color, ls=ls,
                   label=f"{split} {'RGB'[c]}")
ax[1].set(title="Pixel intensity: training slides vs test patients", xlabel="intensity"); ax[1].legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.show()
dist.loc["total"] = dist.sum(); display(dist)

*Observation (write after running):* are the classes balanced? Do the colour distributions of the training slides and the unseen test patients differ? A difference would be domain shift from staining, which motivates the stain-augmentation experiments.

In [ ]:
size = max(SIZES)
arr = load_split(prepare_data(DATA_ROOT, size, download=False, source_dirs=["/kaggle/input"]), "train", in_memory=False)
fig, axes = plt.subplots(3, 9, figsize=(16, 6))
for c, name in enumerate(CLASS_NAMES):
    for r, i in enumerate(rng.choice(np.flatnonzero(arr.labels == c), 3, replace=False)):
        axes[r, c].imshow(arr.images[i]); axes[r, c].axis("off")
    axes[0, c].set_title(f"{name}\n{CLASS_DESCRIPTIONS[name]}", fontsize=8)
plt.suptitle(f"Training patches at {size} px"); plt.tight_layout(); plt.show()

## 6. Pipeline check

In [ ]:
if RUN_CHECKS:
    !python -m pytest -q -p no:cacheprovider tests

In [ ]:
def show_study(out_dir):
    """Render OBSERVATIONS.md stage by stage, each followed by its figure."""
    out_dir = Path(out_dir); obs = out_dir / "OBSERVATIONS.md"
    if not obs.exists():
        print(f"{obs} not found - run the study first"); return
    if (out_dir / "figures/progress.png").exists():
        display(Image(str(out_dir / "figures/progress.png")))
    for section in re.split(r"\n(?=## )", obs.read_text()):
        display(Markdown(re.sub(r"!\[.*?\]\(.*?\)", "", section)))
        m = re.match(r"## (\S+)", section)
        fig = out_dir / "figures" / f"stage_{m.group(1)}.png" if m else None
        if fig is not None and fig.exists():
            display(Image(str(fig)))

## 7. Study 1: CNN hyperparameters from scratch (28 px)
Stages run in order: optimisation → architecture → regularisation. Each stage keeps the setting with the best validation macro F1 and carries it into the next stage. With two GPUs, a stage's settings train in parallel.

For each stage the report gives:
- the hypothesis from theory;
- a table of measurements: F1, convergence epoch, train–validation gap, first-layer gradient norm, time and parameters;
- automatic observations computed from the runs;
- space for your interpretation.

In [ ]:
if RUN_CNN_STUDY:
    seeds = f"--seeds {' '.join(map(str, SEEDS))}" if SEEDS else ""
    !python scripts/run_ablation.py --plan configs/ablation/cnn_ablation.yaml {seeds} --gpus {GPUS} {PLATFORM}
show_study("runs/cnn_ablation")

## 8. Baseline reproduction: ResNet-18 (MedMNIST v2 protocol)
The published result is test accuracy 0.907 and AUC 0.983 at 28 px. Matching it validates the data pipeline, training and evaluation.

In [ ]:
if RUN_B2:
    !python scripts/train.py --config configs/models/b2_resnet18.yaml {PLATFORM} eval.test=true
f = Path("runs/B2_resnet18/seed42/metrics.json")
if f.exists():
    m = json.load(open(f))
    print(f"ResNet-18 (28 px) test accuracy {m['test']['accuracy']:.4f} (published 0.907) | "
          f"macro AUC {m['test']['macro_auc']:.4f} (published 0.983)")

## 9. Study 2: Transfer-learning hyperparameters (224 px)
ImageNet weights are fine-tuned on a stratified 25 % of the training split for 6 epochs per setting; validation uses the full split. Absolute scores are therefore lower than a full run, but every setting is compared on identical terms.

Set `TRANSFER_BASE` to the DeiT-Small config to repeat the study on a vision transformer. Comparing the two answers whether CNNs and transformers want different fine-tuning hyperparameters.

In [ ]:
model_tag = Path(TRANSFER_BASE).stem.split("_", 1)[1]          # e.g. resnet50, deit_small
TRANSFER_OUT = f"runs/transfer_ablation_{model_tag}"
if RUN_TRANSFER_STUDY:
    !python scripts/run_ablation.py --plan configs/ablation/transfer_ablation.yaml --base {TRANSFER_BASE} --output-dir {TRANSFER_OUT} --best-config-out configs/models/{Path(TRANSFER_BASE).stem}_tuned.yaml --gpus {GPUS} {PLATFORM}
show_study(TRANSFER_OUT)

## 10. Study 3: Joint search with Optuna (optional)
One-at-a-time search misses interactions; the best learning rate, for example, depends on the batch size. A TPE sampler with median pruning searches five hyperparameters of the tuned CNN together.

In [ ]:
if RUN_OPTUNA:
    if Path("configs/models/b1_cnn_tuned.yaml").exists():
        !python scripts/tune.py --space configs/tuning/optuna_cnn.yaml --n-trials 20 {PLATFORM}
    else:
        print("Run Study 1 first (it writes configs/models/b1_cnn_tuned.yaml)")
t = Path("runs/optuna/trials.csv")
if t.exists():
    trials = pd.read_csv(t)
    done = trials[trials.state == "COMPLETE"].sort_values("value", ascending=False)
    print(f"{len(done)} completed, {(trials.state == 'PRUNED').sum()} pruned trials")
    display(done.head(8)[["number", "value"] + [c for c in trials if c.startswith("params_")]])
    plt.figure(figsize=(7, 3)); plt.plot(done.sort_values("number").number, done.sort_values("number").value.cummax())
    plt.xlabel("trial"); plt.ylabel("best val macro F1 so far"); plt.title("Optuna search progress"); plt.show()

*Observation:* did the joint search beat the one-at-a-time winner by more than ~0.005 F1? Which parameters moved most from the ablation's choices?

## 11. Study 4: Benchmark on the independent test cohort
Each final model is trained with 3 seeds on the full training split and evaluated once on the 50 unseen patients, with 95 % bootstrap CIs and paired tests against the tuned scratch CNN. Tuned configurations from Studies 1–2 are included automatically.

In [ ]:
if RUN_BENCHMARK:
    only = f"--only {' '.join(BENCHMARK_ONLY)}" if BENCHMARK_ONLY else ""
    !python scripts/run_suite.py --suite configs/suites/e1_e3_benchmark.yaml {only} --gpus {GPUS} {PLATFORM}
if list(Path("runs/e1_e3_benchmark").glob("*/seed*/metrics.json")):
    !python scripts/report.py --suite configs/suites/e1_e3_benchmark.yaml
    rep = Path("reports/e1_e3_benchmark")
    display(Markdown(re.sub(r"!\[.*?\]\(.*?\)", "", (rep / "REPORT.md").read_text())))
    for f in ("accuracy_vs_params.png", "confusion_best.png", "reliability_best.png"):
        if (rep / "figures" / f).exists(): display(Image(str(rep / "figures" / f)))

## 12. Study 5: What does the best model look at?
- **SHAP (expected gradients)** shows, pixel by pixel, what pushed the prediction.
- **Stain-space SHAP** splits that evidence between haematoxylin (nuclei) and eosin (cytoplasm, stroma).
- A **weight-randomisation check** confirms the maps depend on what the model learned.

In [ ]:
summary = Path("reports/e1_e3_benchmark/summary.csv")
if RUN_EXPLAIN and summary.exists():
    best = pd.read_csv(summary).sort_values("test_macro_f1_mean", ascending=False).id.iloc[0]
    RUN = f"runs/e1_e3_benchmark/{best}/seed42"
    print("best model:", best)
    !python scripts/explain.py --run {RUN} --per-class 10
    !python scripts/explain.py --run {RUN} --per-class 3 --stain-space --no-sanity
    if Path("runs/e1_e3_benchmark/B1_cnn_tuned/seed42/best.pt").exists() and best != "B1_cnn_tuned":
        !python scripts/explain.py --run runs/e1_e3_benchmark/B1_cnn_tuned/seed42 --per-class 3
    for f in sorted(Path(RUN, "explain").glob("*.png")):
        print(f.name); display(Image(str(f), width=1000))
    print("randomisation check (Spearman, low = explanations depend on the learned weights):",
          json.load(open(f"{RUN}/explain/sanity_check.json"))["spearman_mean"])

In [ ]:
if RUN_E4:
    !python scripts/run_suite.py --suite configs/suites/e4_color_aug.yaml --gpus {GPUS} {PLATFORM}
    !python scripts/report.py --suite configs/suites/e4_color_aug.yaml
if RUN_E5:
    !python scripts/run_suite.py --suite configs/suites/e5_resolution.yaml --gpus {GPUS} {PLATFORM}
    !python scripts/report.py --suite configs/suites/e5_resolution.yaml

## 13. Observation log, GPU usage and outputs

In [ ]:
gpu_monitor.terminate()
from crc_tissuenet.viz import plot_gpu_log
if Path(GPU_LOG).exists() and Path(GPU_LOG).stat().st_size:
    util = plot_gpu_log(GPU_LOG, "/kaggle/working/gpu_usage.png")
    display(Image("/kaggle/working/gpu_usage.png")); print("mean utilisation per GPU (%):", util)

# One observation log combining every study run so far (download it from the Output tab)
parts = ["# Observation log", "",
         f"GPUs: {', '.join(torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count()))} | "
         f"torch {torch.__version__} | timm {timm.__version__}", ""]
for p in [Path("runs/cnn_ablation/OBSERVATIONS.md"), *sorted(Path("runs").glob("transfer_ablation_*/OBSERVATIONS.md")),
          Path("reports/e1_e3_benchmark/REPORT.md")]:
    if p.exists():
        parts += [p.read_text().replace("](figures/", f"]({p.parent}/figures/"), ""]
Path("/kaggle/working/OBSERVATION_LOG.md").write_text("\n".join(parts))

!find runs -name last.pt -delete 2>/dev/null
bundle = Path("/kaggle/working/results_bundle"); shutil.rmtree(bundle, ignore_errors=True)
for src in [Path("reports"), Path("runs/cnn_ablation"), *Path("runs").glob("transfer_ablation_*"), Path("runs/optuna")]:
    if src.exists():
        shutil.copytree(src, bundle / src.name, ignore=shutil.ignore_patterns("*.pt", "*.npz", "seed*"), dirs_exist_ok=True)
for f in ["/kaggle/working/OBSERVATION_LOG.md", "/kaggle/working/gpu_usage.png"] + [str(p) for p in Path("configs/models").glob("*_tuned.yaml")]:
    if Path(f).exists(): shutil.copy(f, bundle)
shutil.make_archive("/kaggle/working/crc_tissuenet_results", "zip", bundle)
!du -sh /kaggle/working/crc-tissuenet/runs /kaggle/working/crc_tissuenet_results.zip 2>/dev/null
print("Download crc_tissuenet_results.zip from the Output tab.")

### Notes
- **Validation scores are optimistic.** Training and validation patches come from the same slides; only the test cohort measures generalisation to new patients.
- **Single-seed noise.** Differences below about 0.005 macro F1 between single-seed runs should be treated as ties.
- **Research and education only;** not for clinical use.